In [1]:
from pathlib import Path
import sys

project_dir = Path.cwd()

print("Project folder:", project_dir)
print("Python executable:", sys.executable)

if not (project_dir / ".git").is_dir():
    raise RuntimeError("Open this notebook from the repository root.")

evaluation_dir = project_dir / "data" / "evaluation"
evaluation_dir.mkdir(parents=True, exist_ok=True)

print("Evaluation folder ready:", evaluation_dir)

Project folder: C:\Users\Khaleel\workplace-ai-operations-lab
Python executable: C:\Users\Khaleel\workplace-ai-operations-lab\.venv\Scripts\python.exe
Evaluation folder ready: C:\Users\Khaleel\workplace-ai-operations-lab\data\evaluation


In [2]:
import json

# Compact helper for manually labeled development cases.
def make_case(
    case_id, question, behavior, answer,
    answer_passages, diagnostic_passages, reason
):
    return {
        "id": case_id,
        "question": question,
        "expected_behavior": behavior,
        "reference_answer": answer,
        "answer_supporting_passage_ids": answer_passages,
        "diagnostic_passage_ids": diagnostic_passages,
        "reason": reason,
    }


cases = [
    make_case(
        "DEV-001",
        "How many annual leave days do full-time employees receive?",
        "answer",
        "Full-time employees receive 20 days per calendar year.",
        ["leave_policy:1"], [],
        "The policy explicitly states the entitlement."
    ),
    make_case(
        "DEV-002",
        "What is the monthly home internet reimbursement amount?",
        "abstain", None,
        [], ["remote_work_policy:3"],
        "No amount is specified; this does not establish zero reimbursement."
    ),
    make_case(
        "DEV-003",
        "How many annual leave days do contractors receive?",
        "abstain", None,
        [], ["leave_policy:1"],
        "The retrieved entitlement covers full-time employees, not contractors."
    ),
    make_case(
        "DEV-004",
        "What is the yearly PTO allowance for staff?",
        "qualified_answer",
        "For full-time employees, the policy provides 20 days per calendar year.",
        ["leave_policy:1"], [],
        "Assume PTO means annual leave in this lab; preserve the full-time scope."
    ),
    make_case(
        "DEV-005",
        "What is the cafeteria breakfast menu?",
        "abstain", None,
        [], [],
        "The supplied documents do not describe a cafeteria menu."
    ),
    make_case(
        "DEV-006",
        "Where should staff submit annual leave requests?",
        "answer",
        "Annual leave requests must be submitted through the employee portal.",
        ["leave_policy:2"], [],
        "The policy explicitly identifies the submission channel."
    ),
    make_case(
        "DEV-007",
        "Who approves remote work arrangements?",
        "answer",
        "The employee's manager must approve the arrangement.",
        ["remote_work_policy:2"], [],
        "The policy explicitly identifies the approver."
    ),
    make_case(
        "DEV-008",
        "What should staff do if a company laptop is lost?",
        "answer",
        "Report the lost company laptop to the IT service desk immediately.",
        ["it_support_policy:1"], [],
        "The policy explicitly states the required action."
    ),
    make_case(
        "DEV-009",
        "What is the yearly bonus allowance for staff?",
        "abstain", None,
        [], [],
        "No bonus entitlement is established by the supplied documents."
    ),
]

dataset = {
    "schema_version": "1.0",
    "dataset_version": "1.0",
    "split": "development",
    "label_source": "Manually specified and reviewed in this training lab",
    "passage_id_scheme": "Filename stem plus policy-body line number",
    "limitations": [
        "All questions have already been examined during development.",
        "This is not a held-out test set.",
        "Expected behavior is a reference label, not a system prediction.",
        "Diagnostic passages do not supply the requested answer.",
        "Passage IDs must be reviewed if document lines change.",
    ],
    "cases": cases,
}

# Validate labels before saving.
allowed_behaviors = {"answer", "qualified_answer", "abstain"}

assert len({case["id"] for case in cases}) == len(cases)

for case in cases:
    assert case["expected_behavior"] in allowed_behaviors

    if case["expected_behavior"] == "abstain":
        assert case["reference_answer"] is None
        assert not case["answer_supporting_passage_ids"]
    else:
        assert case["reference_answer"]
        assert case["answer_supporting_passage_ids"]

dataset_path = evaluation_dir / "development_v1.json"
serialized = json.dumps(dataset, indent=2, ensure_ascii=False) + "\n"

if dataset_path.exists():
    if dataset_path.read_text(encoding="utf-8") != serialized:
        raise RuntimeError("Existing dataset differs. Review before replacing it.")
    print("Dataset already matches.")
else:
    dataset_path.write_text(serialized, encoding="utf-8")
    print("Dataset created:", dataset_path)

print("Validated cases:", len(cases))

for behavior in sorted(allowed_behaviors):
    count = sum(case["expected_behavior"] == behavior for case in cases)
    print(f"{behavior}: {count}")

Dataset already matches.
Validated cases: 9
abstain: 4
answer: 4
qualified_answer: 1


In [3]:
import re
import json

STOP_WORDS = {
    "a", "an", "the", "is", "are", "do", "does", "how",
    "what", "which", "of", "to", "for", "per", "in",
    "on", "and", "or", "with",
}

QUERY_EXPANSIONS = {
    "pto": {"annual", "leave"},
    "yearly": {"annual"},
    "staff": {"employees"},
}

def tokenize(text):
    return set(re.findall(r"[a-z0-9]+", text.lower())) - STOP_WORDS


passages = []

for file_path in sorted(
    (project_dir / "data" / "synthetic").glob("*.txt")
):
    text = file_path.read_text(encoding="utf-8")
    _, separator, body = text.partition("\n\n")

    if not separator:
        raise ValueError(f"Missing header/body separator: {file_path.name}")

    for line_number, line in enumerate(body.splitlines(), start=1):
        sentence = line.strip()
        if sentence:
            passages.append({
                "source": file_path.name,
                "passage_id": f"{file_path.stem}:{line_number}",
                "text": sentence,
                "tokens": tokenize(sentence),
            })

if not passages:
    raise RuntimeError("No policy passages found.")


def retrieve_for_evaluation(question, *, expanded=False, top_k=2):
    query_tokens = tokenize(question)

    if expanded:
        # Iterate over the original tokens while expanding a separate set.
        original_tokens = set(query_tokens)
        for token in original_tokens:
            query_tokens.update(QUERY_EXPANSIONS.get(token, set()))

    results = []

    for passage in passages:
        matched_words = query_tokens & passage["tokens"]

        if matched_words:
            results.append({
                "source": passage["source"],
                "passage_id": passage["passage_id"],
                "text": passage["text"],
                "score": len(matched_words),
                "matched_words": sorted(matched_words),
            })

    results.sort(key=lambda item: (-item["score"], item["passage_id"]))
    return results[:top_k]


# Evaluate the saved dataset rather than an in-memory draft.
saved_dataset = json.loads(
    (evaluation_dir / "development_v1.json").read_text(encoding="utf-8")
)

available_ids = {passage["passage_id"] for passage in passages}

for case in saved_dataset["cases"]:
    referenced_ids = (
        case["answer_supporting_passage_ids"]
        + case["diagnostic_passage_ids"]
    )
    missing_ids = set(referenced_ids) - available_ids

    if missing_ids:
        raise ValueError(
            f"{case['id']} references missing passages: {sorted(missing_ids)}"
        )

print("Indexed passages:", len(passages))
print("Loaded evaluation cases:", len(saved_dataset["cases"]))
print("All referenced passage IDs exist.")

Indexed passages: 9
Loaded evaluation cases: 9
All referenced passage IDs exist.


In [4]:
evaluation_results = []
evaluation_summary = {}

for method_name, use_expansion in [
    ("baseline", False),
    ("expanded", True),
]:
    answerable_count = 0
    top1_hits = 0
    top2_hits = 0
    abstention_count = 0
    abstention_with_results = 0

    print(f"\nMETHOD: {method_name}")

    for case in saved_dataset["cases"]:
        results = retrieve_for_evaluation(
            case["question"],
            expanded=use_expansion,
            top_k=2,
        )

        retrieved_ids = [item["passage_id"] for item in results]
        supporting_ids = set(case["answer_supporting_passage_ids"])
        is_answerable = case["expected_behavior"] != "abstain"

        if is_answerable:
            answerable_count += 1
            top1_hit = bool(supporting_ids & set(retrieved_ids[:1]))
            top2_hit = bool(supporting_ids & set(retrieved_ids))
            top1_hits += int(top1_hit)
            top2_hits += int(top2_hit)

            print(
                f"{case['id']}: top-1 hit={top1_hit}, "
                f"top-2 hit={top2_hit}"
            )
        else:
            abstention_count += 1
            abstention_with_results += int(bool(results))
            top1_hit = None
            top2_hit = None

            print(
                f"{case['id']}: abstention reference; "
                f"retrieved {len(results)} passage(s)"
            )

        evaluation_results.append({
            "method": method_name,
            "case_id": case["id"],
            "question": case["question"],
            "expected_behavior": case["expected_behavior"],
            "top1_supporting_hit": top1_hit,
            "top2_supporting_hit": top2_hit,
            "retrieved_passages": results,
        })

    evaluation_summary[method_name] = {
        "answerable_cases": answerable_count,
        "top1_supporting_hits": top1_hits,
        "top2_supporting_hits": top2_hits,
        "abstention_cases": abstention_count,
        "abstention_cases_with_retrieval": abstention_with_results,
    }

    print(f"\nSupporting passage at rank 1: {top1_hits}/{answerable_count}")
    print(f"Supporting passage within top 2: {top2_hits}/{answerable_count}")
    print(
        "Abstention cases returning passages: "
        f"{abstention_with_results}/{abstention_count}"
    )


METHOD: baseline
DEV-001: top-1 hit=True, top-2 hit=True
DEV-002: abstention reference; retrieved 1 passage(s)
DEV-003: abstention reference; retrieved 2 passage(s)
DEV-004: top-1 hit=False, top-2 hit=False
DEV-005: abstention reference; retrieved 0 passage(s)
DEV-006: top-1 hit=True, top-2 hit=True
DEV-007: top-1 hit=True, top-2 hit=True
DEV-008: top-1 hit=True, top-2 hit=True
DEV-009: abstention reference; retrieved 0 passage(s)

Supporting passage at rank 1: 4/5
Supporting passage within top 2: 4/5
Abstention cases returning passages: 2/4

METHOD: expanded
DEV-001: top-1 hit=True, top-2 hit=True
DEV-002: abstention reference; retrieved 1 passage(s)
DEV-003: abstention reference; retrieved 2 passage(s)
DEV-004: top-1 hit=True, top-2 hit=True
DEV-005: abstention reference; retrieved 0 passage(s)
DEV-006: top-1 hit=False, top-2 hit=True
DEV-007: top-1 hit=True, top-2 hit=True
DEV-008: top-1 hit=True, top-2 hit=True
DEV-009: abstention reference; retrieved 2 passage(s)

Supporting pass

In [5]:
import hashlib
import json
import platform
import sys
from datetime import datetime, timezone

def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


timestamp = datetime.now(timezone.utc)
dataset_file = evaluation_dir / "development_v1.json"
source_dir = project_dir / "data" / "synthetic"

evaluation_report = {
    "created_at_utc": timestamp.isoformat(),
    "dataset_version": saved_dataset["dataset_version"],
    "split": saved_dataset["split"],
    "dataset_sha256": sha256_file(dataset_file),
    "source_sha256": {
        path.name: sha256_file(path)
        for path in sorted(source_dir.glob("*.txt"))
    },
    "environment": {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
    },
    "retrieval_configuration": {
        "top_k": 2,
        "scoring": "Unique query/passage token overlap",
        "tie_break": "Passage ID ascending",
        "token_pattern": "[a-z0-9]+",
        "stop_words": sorted(STOP_WORDS),
        "query_expansions": {
            key: sorted(values)
            for key, values in QUERY_EXPANSIONS.items()
        },
    },
    "summary": evaluation_summary,
    "results": evaluation_results,
    "limitations": [
        "Nine previously examined development cases.",
        "Five answerable cases, including one qualified-answer case.",
        "Supporting-passage hits do not measure answer correctness.",
        "Retrieval on abstention cases does not measure abstention behavior.",
        "No LLM or automatic answerability component was evaluated.",
    ],
}

evidence_dir = project_dir / "evidence"
evidence_dir.mkdir(exist_ok=True)

evaluation_report_path = evidence_dir / (
    f"retrieval_evaluation_"
    f"{timestamp.strftime('%Y%m%dT%H%M%S%fZ')}.json"
)

with evaluation_report_path.open("x", encoding="utf-8") as file:
    json.dump(evaluation_report, file, indent=2, ensure_ascii=False)

print("Evaluation report saved:", evaluation_report_path)
print("Dataset fingerprint:", evaluation_report["dataset_sha256"])
print("Source documents fingerprinted:", len(evaluation_report["source_sha256"]))
print("Method/case results saved:", len(evaluation_report["results"]))

Evaluation report saved: C:\Users\Khaleel\workplace-ai-operations-lab\evidence\retrieval_evaluation_20261011T162826333098Z.json
Dataset fingerprint: 7163a1f584a7df8806214b263ab7a79e0f25e82ec8ba45e934a04193751d020f
Source documents fingerprinted: 3
Method/case results saved: 18
